In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import logging

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

In [ ]:
logger = tf.get_logger()
logger.setLevel(logging.ERROR)

In [ ]:
dataset, metadata = tfds.load('mnist', as_supervised=True, with_info=True)
train_dataset, test_dataset = dataset['train'], dataset['test']

In [ ]:
class_names = [
'Cero', 'Uno', 'Dos', 'Tres', 'Cuatro', 'Cinco', 'Seis', 'Siete',
'Ocho', 'Nueve'
]

In [ ]:
num_train_examples = metadata.splits['train'].num_examples #60 mil datos train
num_test_examples = metadata.splits['test'].num_examples #10 mil datos test

In [ ]:
#Normalizar: Numeros de 0 a 255, que sean de 0 a 1
def normalize(images, labels):
images = tf.cast(images, tf.float32)
images /= 255
return images, labels

In [ ]:
train_dataset = train_dataset.map(normalize)
test_dataset = test_dataset.map(normalize)

In [ ]:
#Estructura de la red
model = tf.keras.Sequential([
tf.keras.layers.Flatten(input_shape=(28,28,1)), #Capa de entrada
tf.keras.layers.Dense(64, activation=tf.nn.relu), #Capas oculta
tf.keras.layers.Dense(64, activation=tf.nn.relu), #Capas oculta
tf.keras.layers.Dense(10, activation=tf.nn.softmax) #para clasificacion
])

In [ ]:
#Función que compila el modelo
model.compile(
optimizer='adam',
loss='sparse_categorical_crossentropy',
metrics=['accuracy']
)

In [ ]:
#Aprendizaje por lotes de 32 cada lote
BATCHSIZE = 32
train_dataset = train_dataset.repeat().shuffle(num_train_examples).batch(BATCHSIZE)
test_dataset = test_dataset.batch(BATCHSIZE)

In [ ]:
#Realizar el aprendizaje
model.fit(
train_dataset, epochs=5,
steps_per_epoch=math.ceil(num_train_examples/BATCHSIZE)
)

In [ ]:
#Evaluar nuestro modelo ya entrenado, contra el dataset de pruebas
test_loss, test_accuracy = model.evaluate(
test_dataset, steps=math.ceil(num_test_examples/32)
)
#Imprime los resultados de precisión
print("Resultado en las pruebas: ", test_accuracy)

In [ ]:
for test_images, test_labels in test_dataset.take(1):
test_images = test_images.numpy()
test_labels = test_labels.numpy()
predictions = model.predict(test_images)
def plot_image(i, predictions_array, true_labels, images):
predictions_array, true_label, img = predictions_array[i], true_labels[i], images[i]
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.imshow(img[...,0], cmap=plt.cm.binary)
predicted_label = np.argmax(predictions_array)
if predicted_label == true_label:
color = 'blue'
else:
color = 'red'
plt.xlabel("Prediccion: {}".format(class_names[predicted_label]), color=color)
def plot_value_array(i, predictions_array, true_label):
predictions_array, true_label = predictions_array[i], true_label[i]
plt.grid(False)
plt.xticks([])
plt.yticks([])
thisplot = plt.bar(range(10), predictions_array, color="#888888")
plt.ylim([0,1])
predicted_label = np.argmax(predictions_array)
thisplot[predicted_label].set_color('red')
thisplot[true_label].set_color('blue')